# EA2 Unidad 1: Modelos de predicción y clasificación

Este notebook ejecuta el flujo completo de la actividad: carga automática desde Hugging Face, preprocesamiento, regresión lineal, clasificación, interpretación y generación de `resumen_modelos.txt`.

In [1]:
from src.sincronicas.hf_loader import (
    DATASET_ID,
    DATASET_SPLIT,
    DATASET_URL,
    load_dataframe,
    prepare_data,
)
from src.sincronicas.regresion_lineal import train_and_evaluate as train_regression
from src.sincronicas.clasificacion_riesgo import train_and_evaluate as train_classification
from src.sincronicas.reportes import build_summary, write_summary


/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Carga automática y construcción del DataFrame

In [2]:
df_raw = load_dataframe()
print(f"Dataset: {DATASET_ID}")
print(f"Split: {DATASET_SPLIT}")
print(f"Dimensiones: {df_raw.shape}")
print(f"Columnas: {df_raw.columns.tolist()}")
print("Tipos de datos:")
print(df_raw.dtypes.to_string())
print("Muestra:")
display(df_raw.head())


Dataset: scikit-learn/auto-mpg
Split: train
Dimensiones: (398, 9)
Columnas: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin', 'car name']
Tipos de datos:
mpg             float64
cylinders         int64
displacement    float64
horsepower       object
weight            int64
acceleration    float64
model year        int64
origin            int64
car name         object
Muestra:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## 1. Preprocesamiento y división de datos

In [3]:
data = prepare_data(df_raw)
df = data["df"]
print(f"Duplicados antes de limpiar: {data['duplicates_before']}")
print("Nulos antes de imputar:")
print(data["missing_before"][data["missing_before"] > 0].to_string() or "Ninguno")
print(f"Dimensiones después de limpiar: {df.shape}")
print("Distribución de clases:")
print(data["y_cls"].value_counts().sort_index().to_string())
print(f"Regresión: train={len(data['y_train_r'])}, test={len(data['y_test_r'])}")
print(f"Clasificación: train={len(data['y_train_c'])}, test={len(data['y_test_c'])}")
print("Nulos después de imputar: 0")


Duplicados antes de limpiar: 0
Nulos antes de imputar:
horsepower    6
Dimensiones después de limpiar: (398, 9)
Distribución de clases:
efficiency
High    201
Low     197
Regresión: train=318, test=80
Clasificación: train=318, test=80
Nulos después de imputar: 0


/workspaces/ML_2026Q4_2/src/sincronicas/hf_loader.py:32: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["efficiency"] = np.where(df["mpg"] >= median_mpg, "High", "Low")


## 2. Regresión lineal

In [4]:
regression = train_regression(data)
regression_model = regression["model"]
coefficients = regression["coefficients"]
mae = regression["mae"]
rmse = regression["rmse"]
r2 = regression["r2"]
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print("Coeficientes:")
print(coefficients.sort_values(ascending=False).to_string())
print(f"Intercepto: {regression_model.intercept_:.4f}")


MAE: 2.2556
RMSE: 2.8632
R²: 0.8475
Coeficientes:
model year      2.859148
displacement    1.465106
origin          1.073207
acceleration    0.187582
cylinders      -0.264273
horsepower     -0.494947
weight         -5.659110
Intercepto: 23.6082


## 3. Clasificación

In [5]:
classification = train_classification(data)
classification_model = classification["model"]
accuracy = classification["accuracy"]
precision = classification["precision"]
recall = classification["recall"]
f1 = classification["f1"]
confusion = classification["confusion"]
print(f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
display(classification["class_metrics"])
print("Matriz de confusión (filas reales, columnas predichas; [Low, High]):")
print(confusion)


Accuracy: 0.9750 | Precision: 0.9750 | Recall: 0.9750 | F1: 0.9750


,precision,recall,f1,support
Low,0.975,0.975,0.975,40
High,0.975,0.975,0.975,40


Matriz de confusión (filas reales, columnas predichas; [Low, High]):
[[39  1]
 [ 1 39]]


## 4. Interpretación y resumen automático

In [6]:
summary = build_summary(df_raw, data, regression, classification)
write_summary(summary)
print(summary)
print("Archivo generado: resumen_modelos.txt")


RESUMEN AUTOMÁTICO DE MODELOS DE MACHINE LEARNING
Fecha de ejecución UTC: 2026-10-25T16:22:36.776642+00:00
Dataset: scikit-learn/auto-mpg
URL: https://huggingface.co/datasets/scikit-learn/auto-mpg
Split: train
Dimensiones iniciales: (398, 9)
Dimensiones después de limpiar: (398, 9)

PREPROCESAMIENTO
----------------
Nulos iniciales: {'horsepower': 6}
Duplicados iniciales: 0
Se reemplazaron '?' por NaN, se eliminaron nombres de vehículos y duplicados.
Se convirtieron los predictores a numéricos y se imputó la mediana.
Imputadores y escaladores se ajustaron únicamente con train.
La etiqueta efficiency usa la mediana de mpg (23.00): High >= mediana, Low < mediana.
Predictoras: ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model year', 'origin']
División: 80% train / 20% test; random_state=42.

REGRESIÓN LINEAL
----------------
Objetivo: predecir mpg a partir de características físicas y mecánicas.
Algoritmo: LinearRegression con parámetros predeterminados.
MAE: 2.